# 🧭 Nextify Course Notebook – Agent 1 (Idea Intake + Brainstorming)

This notebook mirrors the course reading notebooks while using the **Google ADK**
(Gemini first) and a Switch Agent that can fall back to OpenAI when a token limit
or error is encountered. It is Kaggle-friendly and assumes API keys are provided
through environment variables or the Kaggle sidebar.


## 0. Environment setup (Kaggle-ready)
- Uses Google ADK (`google-genai` / `google-generativeai`) for Gemini calls.
- Switch Agent prompts the subscriber before routing to OpenAI when Gemini hits a limit.
- The Switch Agent stores a running summary so context is preserved across providers.


In [ ]:
# Install dependencies (uncomment if needed on Kaggle)
# !pip install -q google-generativeai google-genai openai python-dotenv


In [ ]:
import os, json, textwrap
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple

# ADK / LLM clients
try:
    from google import genai  # google-genai
    import google.generativeai as genai_legacy  # legacy client still used by ADK wrappers
except Exception:
    genai = None
    genai_legacy = None

try:
    from openai import OpenAI
except Exception:
    OpenAI = None


## 1. Idea Form (user-filled)
Edit the placeholders to reflect your idea. This follows the course template and feeds the brainstorming agents.


In [ ]:
idea_form = {
    "idea_name": "Nextify - Product Strategy Copilot",
    "persona": "Founder building AI-first PM toolkit",
    "problem": "Quickly turn raw ideas into prioritized product snapshots with minimal manual research.",
    "target_users": "Early-stage founders and PMs",
    "solution_approach": "Multi-agent system for intake, brainstorming, market sizing, and snapshot synthesis.",
    "constraints": "Keep outputs concise; note switching provider when Gemini tokens are exhausted.",
}
print(json.dumps(idea_form, indent=2))


## 2. Switch Agent (Gemini → OpenAI) – shared service
- Lives outside the per-agent architecture; every agent can call it.
- Tries Gemini (ADK) first; on token limit/error, asks the subscriber whether to switch.
- When switching, it forwards a continuity summary plus all collected outputs for that agent.
- Tracks the provider chain used so far for observability.


In [ ]:
class LLMError(Exception):
    ...

@dataclass
class LLMConfig:
    primary: str = "gemini"  # 'gemini' or 'openai'
    gemini_model: str = "gemini-1.5-pro-latest"
    openai_model: str = "gpt-4o-mini"

@dataclass
class SwitchState:
    continuity_summary: str = ""
    outputs: List[str] = None
    providers: List[str] = None

    def __post_init__(self):
        self.outputs = self.outputs or []
        self.providers = self.providers or []

class LLMSwitchAgent:
    # Shared router that wraps Gemini (ADK) and OpenAI and preserves context.

    def __init__(self, cfg: LLMConfig, subscriber_allows_switch: bool = True):
        self.cfg = cfg
        self.subscriber_allows_switch = subscriber_allows_switch
        self._gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY")) if genai else None
        self._openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) if OpenAI else None

    def _call_gemini(self, prompt: str) -> str:
        if not self._gemini_client:
            raise LLMError("Gemini client not available")
        resp = self._gemini_client.models.generate_content(
            model=self.cfg.gemini_model,
            contents=prompt,
        )
        return resp.text

    def _call_openai(self, prompt: str) -> str:
        if not self._openai_client:
            raise LLMError("OpenAI client not available")
        resp = self._openai_client.chat.completions.create(
            model=self.cfg.openai_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        return resp.choices[0].message.content

    def run(self, prompt: str, state: SwitchState):
        """Try Gemini first; if a limit/error occurs and subscriber consents, switch to OpenAI."""
        try:
            text = self._call_gemini(prompt)
            state.providers.append("gemini")
            return text, state
        except Exception as e:
            state.providers.append("gemini:error")
            if not self.subscriber_allows_switch:
                raise LLMError(f"Gemini failed and switch not allowed: {e}")
            try:
                prompt_with_summary = textwrap.dedent(f"""
                Previous summary: {state.continuity_summary or 'N/A'}
                Prior outputs: {json.dumps(state.outputs[-3:], ensure_ascii=False)}
                Continue: {prompt}
                """)
                text = self._call_openai(prompt_with_summary)
                state.providers.append("openai")
                return text, state
            except Exception as e2:
                state.providers.append("openai:error")
                raise LLMError(f"Both providers failed: {e} | {e2}")

    def update_summary(self, state: SwitchState, new_output: str) -> SwitchState:
        state.outputs.append(new_output)
        if len(state.outputs) > 1:
            state.continuity_summary = " | ".join(state.outputs[-3:])
        else:
            state.continuity_summary = new_output[:400]
        return state


## 3. Brainstorming sub-agents (ADK style)
- **MarketAnalysisSubAgent**: TAM/SAM/SOM + market signals for the exact idea.
- **CrazyIdeaSubAgent**: playful cross-industry mashups that consider trends.
- **SynthesisSubAgent**: merges both, scores candidates, and proposes evaluated options.
Each sub-agent runs through the Switch Agent so they inherit continuity and can recover on provider limits.


In [ ]:
def market_analysis_prompt(idea: Dict[str, Any]) -> str:
    return textwrap.dedent(f"""
    You are MarketAnalysisSubAgent. Build a concise TAM/SAM/SOM breakdown for '{idea.get('idea_name')}'.
    Include: target users, assumptions, pricing anchor, TAM/SAM/SOM numbers (rough), top risks, and demand signals.
    Return markdown with bullet lists and a short table.
    """)


def crazy_idea_prompt(idea: Dict[str, Any]) -> str:
    return textwrap.dedent(f"""
    You are CrazyIdeaSubAgent. Make bold cross-industry combinations for '{idea.get('idea_name')}'.
    Blend at least two adjacent or surprising industries, and ground them with current tech/industry trends.
    Return 3 concepts with hooks and differentiation; keep feasibility notes.
    """)


def synthesis_prompt(idea: Dict[str, Any], market: str, wild: str) -> str:
    return textwrap.dedent(f"""
    You are SynthesisSubAgent. Combine the market analysis and the wild ideas to propose 3-5 evaluated concepts.
    For each concept provide: one-liner, user value, feasibility, risk, and an overall score (1-5).
    End with a short Product Snapshot draft that extends the Idea Form (fields: Problem, Users, Solution, Differentiation, TAM/SAM/SOM, Risks, Provider chain used).
    Use concise markdown.
    """)


## 4. Brainstorming orchestrator (with revision loop)
- Runs the three sub-agents through the Switch Agent.
- Maintains continuity summary and provider chain for transparency.
- Allows up to two revisions with user feedback; final output is a Product Snapshot to hand off downstream.


In [ ]:
def run_brainstorming(
    idea: Dict[str, Any],
    switch_agent: LLMSwitchAgent,
    state: SwitchState,
    user_feedback: str = "",
    revision_count: int = 0,
):
    market_out, state = switch_agent.run(market_analysis_prompt(idea), state)
    state = switch_agent.update_summary(state, market_out)

    crazy_out, state = switch_agent.run(crazy_idea_prompt(idea), state)
    state = switch_agent.update_summary(state, crazy_out)

    synth_prompt = synthesis_prompt(idea, market_out, crazy_out)
    if user_feedback:
        synth_prompt += f"
Incorporate this subscriber feedback: {user_feedback}"
    synth_out, state = switch_agent.run(synth_prompt, state)
    state = switch_agent.update_summary(state, synth_out)

    product_snapshot = synth_out
    revision_count = min(revision_count, 2)
    return product_snapshot, state.providers, state.continuity_summary, revision_count, state


## 5. First pass (no feedback yet)
Run this cell to generate the initial Product Snapshot. If Gemini hits a limit, the Switch Agent will prompt the subscriber to approve an OpenAI retry and will pass along the running summary.


In [ ]:
cfg = LLMConfig(primary="gemini")
switch_agent = LLMSwitchAgent(cfg, subscriber_allows_switch=True)
state = SwitchState()

product_snapshot, providers_used, continuity_summary, revision_count, state = run_brainstorming(
    idea_form,
    switch_agent,
    state,
    user_feedback="",
    revision_count=0,
)

print("Providers used:", providers_used)
print("Continuity summary (truncated):", continuity_summary[:400])
print("
--- Product Snapshot ---
")
print(product_snapshot)


## 6. User confirmation loop (max 2 revisions)
1. Set `user_feedback` below (choose a concept, add constraints, or request a tweak).
2. Re-run the cell to iterate. The Switch Agent retains summaries and provider chain across revisions.


In [ ]:
user_feedback = ""  # e.g., "Pick concept C2, emphasize onboarding risk, shorten TAM notes"
revision_count += 1
product_snapshot, providers_used, continuity_summary, revision_count, state = run_brainstorming(
    idea_form,
    switch_agent,
    state,
    user_feedback=user_feedback,
    revision_count=revision_count,
)

print("Revision:", revision_count)
print("Providers used:", providers_used)
print("Continuity summary (truncated):", continuity_summary[:400])
print("
--- Product Snapshot ---
")
print(product_snapshot)
